In [1]:
import pandas as pd
from datasets import load_dataset, Dataset

/opt/conda/envs/afrimmd/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
preds_df = pd.read_csv("finetune/test_predictions.csv", index_col=0)
preds_df.head()

,predictions,references,language,candidates
id,,,,
0,"[256051, 14734, 2442, 29213, 88317, 2790, 2430...",There are three girls with head scarves in fro...,eng_Latn,There are three girls with heads scarves under...
1,"[256025, 71112, 1022, 8423, 248272, 62, 14405,...",Mũndũ ũrarĩithia gĩcagi nĩ ahurũkaga thĩ kũger...,kik_Latn,Mũndũ ũrarũ na skithagi kĩa aurũkaga rũ yagere...
2,"[256025, 6433, 83541, 761, 711, 11126, 16737, ...",Omusajja ayambadde sikaati emmyuufu ng'alinnya...,lug_Latn,Omusajja ayambadde sikaati emmyuufu ng'alinnya...
3,"[256025, 61493, 80, 35, 248116, 75223, 188, 35...",Imbwa y'umukara yambaye ishati y'ibara ry'umuh...,kin_Latn,Imbwa y'umukara yambaye ishati y'umuh ry'umuho...
4,"[256073, 22435, 63970, 7300, 219, 359, 133, 22...",Ŋutsu aɖe si do tavu dzĩ la le tɔɖim ɖe agakpe...,ewe_Latn,Ŋutsu aɖe si do ta dz dzĩ la nɔ agakɖim ɖe aga...


In [15]:
preds_df["language"].value_counts()

language
fuv_Latn    441
ibo_Latn    434
amh_Ethi    427
kab_Latn    423
hau_Latn    416
eng_Latn    413
ewe_Latn    413
lua_Latn    412
lug_Latn    410
kmb_Latn    409
afr_Latn    407
bem_Latn    406
kik_Latn    406
dyu_Latn    399
yor_Latn    393
cjk_Latn    393
kam_Latn    390
lin_Latn    385
kon_Latn    378
dik_Latn    374
kin_Latn    367
Name: count, dtype: int64

In [9]:
full_df  = load_dataset("AfriMM/AFRICaption")["train"]

In [16]:
df = full_df.to_pandas()

In [17]:
african_langs = ['afr', 'amh', 'bem', 'cjk', 'dik', 'dyu', 'ewe', 'fuv', 'hau', 'ibo', 'kik', 'kab', 'kam', 'kon', 'kmb', 'lua', 'lug', 'lin', 'kin', 'yor']

flat_df = df.melt(
id_vars=["id", "image_id", "eng"],
value_vars=african_langs,
var_name="language",
value_name="caption"
)

In [18]:
flat_df.rename(columns={"eng": "eng_caption"}, inplace=True)

In [19]:
flat_df

,id,image_id,eng_caption,language,caption
0,0,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .,afr,'n Dogtertjie wat in 'n hout speelhuis klim .
1,1,1001773457_577c3a7d70.jpg,A black dog and a tri-colored dog playing with...,afr,'n Swart hond en 'n driekleurige hond wat op d...
2,2,1002674143_1b742ab4b8.jpg,A little girl covered in paint sits in front o...,afr,'n Dogtertjie bedek met verf sit voor 'n gever...
3,3,1003163366_44323f5815.jpg,A man lays on a bench while his dog sits by him .,afr,'N Man lê op 'n bankie terwyl sy hond by hom sit.
4,4,1007129816_e794419615.jpg,A man wears an orange hat and glasses .,afr,'N Man dra 'n oranje hoed en bril .
...,...,...,...,...,...
161815,8086,990890291_afc72be141.jpg,A man is doing a wheelie on a mountain bike .,yor,Ọkùnrin kan ń ṣe kẹ̀kẹ́ kẹ̀kẹ́ lórí kẹ̀kẹ́ òkè .
161816,8087,99171998_7cc800ceef.jpg,A group of people sit atop a snowy mountain .,yor,Àwùjọ àwọn èèyàn kan jókòó lórí òkè tí yìnyín ...
161817,8088,99679241_adc853a5c0.jpg,A tall bird is standing on the sand beside the...,yor,Ẹyẹ gíga kan dúró lórí iyanrìn lẹ́gbẹ̀ẹ́ òkun .
161818,8089,997338199_7343367d7f.jpg,A woman standing near a decorated wall writes .,yor,Obìnrin kan tó dúró lẹ́gbẹ̀ẹ́ ògiri tí wọ́n ṣe...


In [21]:
# Create a new test set by matching reference captions
matched_test_df = flat_df[flat_df["caption"].isin(preds_df["references"])].copy()

# Optional: Reset index and inspect
matched_test_df.reset_index(drop=True, inplace=True)

# Save the matched test set
# matched_test_df.to_csv("reconstructed_test_set.csv", index=False)


In [22]:
matched_test_df.head()

,id,image_id,eng_caption,language,caption
0,33,1042590306_95dea0916c.jpg,Asian man and blond woman holding hands outdoo...,afr,Asiatiese man en blonde vrou wat hande buite h...
1,51,1067675215_7336a694d6.jpg,A man in blue shorts is laying in the street .,afr,'N Man in 'n blou kortbroek lê in die straat.
2,52,1067790824_f3cc97239b.jpg,Two large dogs chasing each other at the beach .,afr,Twee groot honde wat mekaar op die strand jaag.
3,54,107318069_e9f2ef32de.jpg,A crowd watching air balloons at night .,afr,'N Skare wat snags lugballonne kyk.
4,63,1082252566_8c79beef93.jpg,Three dogs of various sizes .,afr,Drie honde van verskillende groottes .


In [23]:
matched_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8751 entries, 0 to 8750
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           8751 non-null   int64 
 1   image_id     8751 non-null   object
 2   eng_caption  8751 non-null   object
 3   language     8751 non-null   object
 4   caption      8751 non-null   object
dtypes: int64(1), object(4)
memory usage: 342.0+ KB


In [27]:
matched_test_df.drop(columns=["id"], inplace=True)


In [28]:
matched_test_df.to_csv("test_data.csv", index=False)